# Fine-tune YOLO26l on the unified defect dataset

Resumes from a previous fine-tune checkpoint (e.g. `best (1).pt`) and trains again with **pseudo-label augmentation** to fix the incomplete-labels problem in the source datasets.

## Running on Kaggle (primary target, 2× T4 16 GB)

1. New Kaggle notebook → Settings → **Accelerator: GPU T4 × 2**, **Internet: On**.
2. Upload your latest `best.pt` as a Kaggle dataset and attach it — the notebook finds it anywhere under `/kaggle/input` via `rglob`. This is the **only** file you need to upload.
3. **Roboflow API key** — either:
   - Add it as a Kaggle Secret named `ROBOFLOW_API_KEY` (Add-ons → Secrets), or
   - Paste it when prompted by `getpass` in Cell 4.
4. The 4 datasets (d1, d2, d4, d5 — d6 was dropped) download automatically from Roboflow at the same pinned versions used originally.
5. Run all cells. Pseudo-labeling adds ~30 min before training; training itself is ~4–5 hrs at 50 epochs.

## Running locally (T1000 4 GB)

Cells 1–7.5 are fine locally (build unified dataset, load checkpoint, baseline eval, pseudo-labeling). Cell 8 (training) will OOM on 4 GB — run it on Kaggle.

## Strategy

- **Drop d6** (`mould-detection-Aaron`) — corrupt-coordinate labels were poisoning training. Mold now comes from d1 only.
- **Pseudo-labeling (Cell 7.5)** — the previous model's recall is low because public-dataset labels are incomplete (visible defects unannotated → treated as background → model penalized for finding them). Use the current model at conf≥0.40 to find likely defects in train+val and ADD them as new labels (only where they don't overlap existing GT, only for classes the source dataset originally had). Test split is untouched for honest eval.
- `lr0=0.001` — 10× lower than default, appropriate for fine-tuning.
- `cls=1.0` — 2× default classification-loss gain.
- `close_mosaic=10`, `mosaic=0.8`, `mixup=0.1` — pulled back so the back half of training sees clean images.
- Baseline eval *before* training so improvement is measurable.

In [14]:
# Cell 1 — Install + imports
import os, sys
IS_KAGGLE = os.path.exists("/kaggle")

if IS_KAGGLE:
    !pip install -q ultralytics roboflow pyyaml

import shutil
import random
from collections import defaultdict
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

print(f"Python : {sys.version.split()[0]}")
print(f"Torch  : {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU    : {gpu_name} ({gpu_vram:.1f} GB)")
else:
    print("!! No GPU detected — training will be impractically slow.")

Python : 3.12.12
Torch  : 2.10.0+cu128
CUDA   : True
GPU    : Tesla T4 (15.6 GB)


In [25]:
# Cell 2 — Environment detection + config
if IS_KAGGLE:
    ROOT = Path("/kaggle/working")
    RAW_DIR = ROOT / "raw_datasets"              # Roboflow downloads go here
    CHECKPOINT_PATH = "/kaggle/input/models/abdelrahmanmajed/2/pytorch/default/1/best (1).pt"
    CACHE = "ram"
    WORKERS = 4
else:
    LOCAL_ROOT = Path(r"D:/Graduation project")
    ROOT = LOCAL_ROOT / "temp" / "train_run"
    RAW_DIR = LOCAL_ROOT / "ML" / "Datasets"     # read existing local folders
    CHECKPOINT_PATH = LOCAL_ROOT / "ML" / "after training on lighting ai" / "best.pt"
    CACHE = "disk"
    WORKERS = 2

UNIFIED = ROOT / "unified_dataset"
RUNS_DIR = ROOT / "runs"
for p in (ROOT, UNIFIED, RUNS_DIR, RAW_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Device: list form triggers ultralytics DDP when >1 GPU is present
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
if N_GPUS >= 2:
    DEVICE = list(range(N_GPUS))          # e.g. [0, 1] for Kaggle 2× T4
elif N_GPUS == 1:
    DEVICE = 0
else:
    DEVICE = "cpu"

# --- Training hyperparameters (fine-tune from epoch-7 yolo26l) ---
# BATCH is the GLOBAL batch across all GPUs; ultralytics splits it per-GPU.
# 8 per T4 is the safe bound for yolo26l at imgsz=640, so 16 total on 2× T4.
EPOCHS              = 35
BATCH               = 32 if N_GPUS >= 2 else 8
IMGSZ               = 640
LR0                 = 0.001   # 10x lower than default 0.01
CLS_GAIN            = 1.0     # 2x default 0.5 — bump classification loss
CLOSE_MOSAIC_EPOCHS = 10
MOSAIC              = 0.8
MIXUP               = 0.1
WARMUP              = 2
PATIENCE            = 15
SEED                = 42

random.seed(SEED)

print(f"Environment    : {'Kaggle' if IS_KAGGLE else 'Local laptop'}")
print(f"GPUs detected  : {N_GPUS}")
print(f"DEVICE         : {DEVICE}")
print(f"Global BATCH   : {BATCH}  ({BATCH // max(N_GPUS, 1)} per GPU)")
print(f"RAW_DIR        : {RAW_DIR}")
print(f"CHECKPOINT_PATH: {CHECKPOINT_PATH}")
print(f"ROOT           : {ROOT}")
print(f"Unified dir    : {UNIFIED}")

# VRAM guard: yolo26l needs ~10 GB per-GPU at per-GPU-batch=8
if torch.cuda.is_available():
    min_vram = min(
        torch.cuda.get_device_properties(i).total_memory / 1e9
        for i in range(N_GPUS)
    )
    if min_vram < 10:
        print(
            f"\n!! WARNING: smallest GPU has only {min_vram:.1f} GB VRAM. "
            f"yolo26l will OOM at per-GPU batch={BATCH // max(N_GPUS, 1)}.\n"
            "   Run training on Kaggle (2× T4 16 GB), or drop BATCH."
        )

Environment    : Kaggle
GPUs detected  : 2
DEVICE         : [0, 1]
Global BATCH   : 32  (16 per GPU)
RAW_DIR        : /kaggle/working/raw_datasets
CHECKPOINT_PATH: /kaggle/input/models/abdelrahmanmajed/2/pytorch/default/1/best (1).pt
ROOT           : /kaggle/working
Unified dir    : /kaggle/working/unified_dataset


In [26]:
# Cell 3 — Dataset specs + unified class config
# d6 (mould-detection-Aaron) was DROPPED from training: ~150+ corrupt-coordinate
# labels and textural mold annotations the model could not learn cleanly. Mold
# is now sourced solely from d1 (Wall Defects Final), which has a clean `mold`
# class. Expect lower total mold instance count but cleaner gradients.
DATASET_SPECS = [
    # (short_id, local_folder_name,                     roboflow_workspace,        roboflow_project,              version)
    ("d1", "Wall Defects Final.v3i.yolo26",            "israas-workspace-icijh",  "wall-defects-final",           3),
    ("d2", "wall crack detection.v4i.yolo26",          "tatung-university-wmt7l", "wall-crack-detection-l0iit",   4),
    ("d4", "water leakage.v1i.yolo26",                 "water-leak-hwh4x",        "water-leakage-ayy4s",          1),
    ("d5", "electrical hazards.v2i.yolo26",            "electrical-hazard-zcack", "electrical-hazards",           2),
]

UNIFIED_CLASSES = ["crack", "water_leakage", "electrical_fault", "mold"]
CLASS_TO_IDX = {c: i for i, c in enumerate(UNIFIED_CLASSES)}

REMAP = {
    "crack":            ["crack", "wall-crack", "major crack", "minor crack",
                         "stairstep_crack", "large", "small", "2"],
    "water_leakage":    ["standing water", "water_seepage", "stain", "leakage", "moisture"],
    "electrical_fault": ["damage wire", "burned socket", "overloaded socket"],
    "mold":             ["mould", "mold"],
}

ORIG_TO_UNIFIED = {}
for unified, originals in REMAP.items():
    for orig in originals:
        ORIG_TO_UNIFIED[orig.strip().lower()] = CLASS_TO_IDX[unified]

print(f"Unified classes: {UNIFIED_CLASSES}")


Unified classes: ['crack', 'water_leakage', 'electrical_fault', 'mold']


In [27]:
# Cell 4 — Acquire datasets (Roboflow download on Kaggle, local folders on laptop)
dataset_paths = {}  # short_id -> Path containing train/, valid/, test/, data.yaml

if IS_KAGGLE:
    # Resolve API key: env var, then Kaggle secret, then getpass
    api_key = ""
    if not api_key:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
            print("Loaded Roboflow API key from Kaggle Secret.")
        except Exception:
            import getpass
            api_key = getpass.getpass("Enter Roboflow API key: ")

    from roboflow import Roboflow
    rf = Roboflow(api_key=api_key)

    for short_id, _local_name, workspace, project_name, version in DATASET_SPECS:
        target = RAW_DIR / short_id
        if (target / "data.yaml").exists():
            print(f"[{short_id}] already at {target}, skipping download.")
            dataset_paths[short_id] = target
            continue
        print(f"[{short_id}] downloading {workspace}/{project_name}/v{version} ...")
        try:
            project = rf.workspace(workspace).project(project_name)
            try:
                ver = project.version(version)
            except RuntimeError:
                versions = project.get_version_information()
                latest = versions[-1]["id"].split("/")[-1]
                print(f"    v{version} not found — falling back to latest v{latest}")
                ver = project.version(int(latest))
            ver.download("yolov8", location=str(target))
            dataset_paths[short_id] = target
            for split in ("train", "valid", "test"):
                img_dir = target / split / "images"
                n = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
                print(f"    {split}: {n} images")
        except Exception as e:
            print(f"  !! FAILED [{short_id}]: {e}")
            print(f"     Skipping — will not be included in merged dataset.")
else:
    for short_id, local_name, *_ in DATASET_SPECS:
        target = RAW_DIR / local_name
        if not (target / "data.yaml").exists():
            print(f"[{short_id}] !! missing {target / 'data.yaml'} — skipping.")
            continue
        dataset_paths[short_id] = target
        print(f"[{short_id}] using local {target}")

print(f"\nResolved {len(dataset_paths)} / {len(DATASET_SPECS)} dataset paths.")

[d1] already at /kaggle/working/raw_datasets/d1, skipping download.
[d2] already at /kaggle/working/raw_datasets/d2, skipping download.
[d4] already at /kaggle/working/raw_datasets/d4, skipping download.
[d5] already at /kaggle/working/raw_datasets/d5, skipping download.

Resolved 4 / 4 dataset paths.


In [28]:
# Cell 5 — Merge all 3 splits into UNIFIED/ with remapped class IDs
SPLIT_MAP = {"train": "train", "valid": "val", "test": "test"}

already_built = (UNIFIED / "images" / "train").exists() and any((UNIFIED / "images" / "train").iterdir())
if already_built:
    print("unified_dataset/ already populated — skipping rebuild.")
else:
    for dst_split in SPLIT_MAP.values():
        (UNIFIED / "images" / dst_split).mkdir(parents=True, exist_ok=True)
        (UNIFIED / "labels" / dst_split).mkdir(parents=True, exist_ok=True)

    merge_stats = defaultdict(lambda: defaultdict(int))

    for short_id, ds_root in dataset_paths.items():
        with open(ds_root / "data.yaml") as f:
            meta = yaml.safe_load(f)
        classes = meta["names"]
        if isinstance(classes, dict):
            classes = [classes[i] for i in sorted(classes.keys())]

        for src_split, dst_split in SPLIT_MAP.items():
            src_img = ds_root / src_split / "images"
            src_lbl = ds_root / src_split / "labels"
            if not src_img.exists():
                continue
            dst_img = UNIFIED / "images" / dst_split
            dst_lbl = UNIFIED / "labels" / dst_split

            for img_path in src_img.iterdir():
                if not img_path.is_file():
                    continue
                new_name = f"{short_id}_{img_path.name}"
                shutil.copy2(img_path, dst_img / new_name)

                new_lines = []
                lbl_src_path = src_lbl / f"{img_path.stem}.txt"
                if lbl_src_path.exists():
                    with open(lbl_src_path) as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) < 5:
                                continue
                            try:
                                old_id = int(float(parts[0]))
                            except ValueError:
                                continue
                            if old_id < 0 or old_id >= len(classes):
                                continue
                            orig_name = classes[old_id].strip().lower()
                            if orig_name not in ORIG_TO_UNIFIED:
                                continue
                            new_id = ORIG_TO_UNIFIED[orig_name]
                            new_lines.append(f"{new_id} {' '.join(parts[1:5])}")
                            merge_stats[dst_split][UNIFIED_CLASSES[new_id]] += 1
                with open(dst_lbl / f"{Path(new_name).stem}.txt", "w") as f:
                    f.write("\n".join(new_lines))
        print(f"[{short_id}] merged from {ds_root.name}")

    for split in SPLIT_MAP.values():
        n_img = len(list((UNIFIED / "images" / split).glob("*")))
        print(f"  {split:>5s}: {n_img} images | classes = {dict(merge_stats[split])}")

data_yaml = {
    "path":  str(UNIFIED).replace("\\", "/"),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc":    len(UNIFIED_CLASSES),
    "names": UNIFIED_CLASSES,
}
with open(UNIFIED / "data.yaml", "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)
print(f"\nWrote {UNIFIED / 'data.yaml'}")

unified_dataset/ already populated — skipping rebuild.

Wrote /kaggle/working/unified_dataset/data.yaml


In [36]:
# Cell 6 — Load starting weights
if CHECKPOINT_PATH and Path(CHECKPOINT_PATH).exists():
    print(f"Starting from checkpoint: {CHECKPOINT_PATH}")
    model = YOLO(str(CHECKPOINT_PATH))
else:
    print("No checkpoint found — starting from COCO-pretrained yolo26l.pt")
    print("(Upload epoch-7 best.pt as a Kaggle dataset and re-run to resume.)")
    model = YOLO("yolo26l.pt")

print(f"Model classes: {model.model.names}")

Starting from checkpoint: /kaggle/input/models/abdelrahmanmajed/2/pytorch/default/1/best (1).pt
Model classes: {0: 'crack', 1: 'water_leakage', 2: 'electrical_fault', 3: 'mold'}


In [30]:
# Cell 7 — Baseline eval BEFORE training (to measure the delta later)
baseline = model.val(
    data=str(UNIFIED / "data.yaml"),
    split="test",
    imgsz=IMGSZ,
    device=DEVICE,
    plots=False,
    save_json=False,
    verbose=False,
)
baseline_per_class = {UNIFIED_CLASSES[i]: float(baseline.box.maps[i]) for i in range(len(UNIFIED_CLASSES))}
baseline_map50 = float(baseline.box.map50)
baseline_map = float(baseline.box.map)

print(f"\nBASELINE (starting checkpoint on test split):")
print(f"  mAP50    : {baseline_map50:.4f}")
print(f"  mAP50-95 : {baseline_map:.4f}")
for cname, ap in baseline_per_class.items():
    print(f"    {cname:<20s} {ap:.4f}")

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
YOLO26l summary (fused): 190 layers, 24,748,824 parameters, 0 gradients, 86.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1185.6±651.8 MB/s, size: 55.5 KB)
val: Scanning /kaggle/working/unified_dataset/labels/test... 3556 images, 1898 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3556/3556 1.5Kit/s 2.4s<0.0s
val: New cache created: /kaggle/working/unified_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 223/223 2.4it/s 1:33<0.5ss
                   all       3556       3106      0.767      0.645      0.683      0.459
Speed: 0.6ms preprocess, 24.3ms inference, 0.0ms loss, 0.1ms postprocess per image

BASELINE (starting checkpoint on test split):
  mAP50    : 0.6830
  mAP50-95 : 0.4586
    crack                0.3059
    water_leaka

In [31]:
# Cell 7.5 — Pseudo-label augmentation (fixes "incomplete labels" problem)
#
# Public detection datasets often have visible defects that aren't annotated.
# YOLO trains "anything outside a labeled box = background", which actively
# pushes the model away from finding the real-but-unlabeled stuff. Result:
# high precision, low recall (exactly what we see — mold R=0.31).
#
# Self-training / pseudo-labeling: use the current model to find likely-defects
# in train+val, ADD them as new labels where they don't overlap existing ones,
# then retrain. The retrained model sees a more complete picture and stops
# being penalized for finding things that are actually there.
#
# Safeguards:
#   1. PSEUDO_CONF=0.40 — only accept very confident predictions
#   2. PSEUDO_IOU=0.30 — only ADD non-overlapping boxes, never replace GT
#   3. Class-scoped to source dataset (d2 only adds 'crack', etc.) — prevents
#      the model from hallucinating water_leakage on a crack-only dataset
#   4. Test split is left untouched for honest evaluation
#
# Adds ~30 min of inference on Kaggle 2× T4 for ~50k images.

PSEUDO_CONF = 0.40
PSEUDO_IOU  = 0.30

# Build per-dataset valid-class set from each data.yaml + REMAP.
# Pseudo-labels for a dataset are restricted to classes that dataset originally
# contained — this is the most important safeguard against cross-class noise.
SOURCE_VALID_CLASSES = {}  # short_id -> set[int] of unified class IDs
for short_id, ds_root in dataset_paths.items():
    yaml_p = ds_root / "data.yaml"
    if not yaml_p.exists():
        SOURCE_VALID_CLASSES[short_id] = set()
        continue
    with open(yaml_p) as f:
        meta = yaml.safe_load(f) or {}
    classes = meta.get("names") or []
    if isinstance(classes, dict):
        classes = [classes[i] for i in sorted(classes.keys())]
    valid = set()
    for c in classes:
        u = ORIG_TO_UNIFIED.get(str(c).strip().lower())
        if u is not None:
            valid.add(u)
    SOURCE_VALID_CLASSES[short_id] = valid

print("Per-dataset valid pseudo-classes (unified IDs / names):")
for sid in sorted(SOURCE_VALID_CLASSES):
    ids = sorted(SOURCE_VALID_CLASSES[sid])
    names = [UNIFIED_CLASSES[i] for i in ids]
    print(f"  {sid}: {ids}  {names}")

def _iou(a, b):
    xa, ya = max(a[0], b[0]), max(a[1], b[1])
    xb, yb = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, xb - xa), max(0.0, yb - ya)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    union = (a[2]-a[0]) * (a[3]-a[1]) + (b[2]-b[0]) * (b[3]-b[1]) - inter
    return inter / union if union > 0 else 0.0

added = defaultdict(lambda: defaultdict(int))   # split -> class -> count
total_processed = 0

for split in ("train", "val"):
    img_dir = UNIFIED / "images" / split
    lbl_dir = UNIFIED / "labels" / split
    if not img_dir.exists():
        print(f"[{split}] no images dir; skipping.")
        continue

    img_paths = sorted(p for p in img_dir.iterdir() if p.is_file())
    print(f"\n[{split}] pseudo-labeling {len(img_paths)} images at conf>={PSEUDO_CONF} ...")

    BATCH_INFER = 64
    for i in range(0, len(img_paths), BATCH_INFER):
        chunk = img_paths[i:i + BATCH_INFER]
        results = model.predict(
            source=[str(p) for p in chunk],
            conf=PSEUDO_CONF,
            imgsz=IMGSZ,
            device=DEVICE,
            verbose=False,
            save=False,
        )
        for p, r in zip(chunk, results):
            sid = p.name.split("_", 1)[0]
            valid = SOURCE_VALID_CLASSES.get(sid, set())
            if not valid:
                continue
            h_img, w_img = r.orig_shape
            lbl_path = lbl_dir / f"{p.stem}.txt"

            # Read existing GT, keep it as-is, also build xyxy list for IoU
            existing_xyxy = []
            existing_lines = []
            if lbl_path.exists():
                with open(lbl_path) as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        parts = line.split()
                        if len(parts) < 5:
                            continue
                        try:
                            cx, cy, bw, bh = map(float, parts[1:5])
                        except ValueError:
                            continue
                        existing_lines.append(line)
                        x1 = (cx - bw / 2) * w_img
                        y1 = (cy - bh / 2) * h_img
                        x2 = (cx + bw / 2) * w_img
                        y2 = (cy + bh / 2) * h_img
                        existing_xyxy.append((x1, y1, x2, y2))

            new_lines = list(existing_lines)
            if r.boxes is not None and len(r.boxes) > 0:
                pred_xyxy = r.boxes.xyxy.cpu().numpy()
                pred_cls  = r.boxes.cls.cpu().numpy().astype(int)
                for j in range(len(pred_xyxy)):
                    pcls = int(pred_cls[j])
                    if pcls not in valid:
                        continue  # cross-class hallucination guard
                    pbox = (float(pred_xyxy[j][0]), float(pred_xyxy[j][1]),
                            float(pred_xyxy[j][2]), float(pred_xyxy[j][3]))
                    # Skip if overlaps any existing GT (keep human label as-is)
                    if any(_iou(pbox, gt) > PSEUDO_IOU for gt in existing_xyxy):
                        continue
                    px1, py1, px2, py2 = pbox
                    cx = ((px1 + px2) / 2) / w_img
                    cy = ((py1 + py2) / 2) / h_img
                    bw = (px2 - px1) / w_img
                    bh = (py2 - py1) / h_img
                    cx = max(0.0, min(1.0, cx)); cy = max(0.0, min(1.0, cy))
                    bw = max(0.0, min(1.0, bw)); bh = max(0.0, min(1.0, bh))
                    if bw < 0.005 or bh < 0.005:
                        continue
                    new_lines.append(f"{pcls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                    added[split][UNIFIED_CLASSES[pcls]] += 1

            with open(lbl_path, "w") as f:
                f.write("\n".join(new_lines))
            total_processed += 1

        done = i + len(chunk)
        if (i // BATCH_INFER) % 25 == 0 or done == len(img_paths):
            print(f"  {done}/{len(img_paths)}  added so far: {dict(added[split])}")

# Invalidate ultralytics label cache so training rescans with the new labels.
for split in ("train", "val"):
    cache = UNIFIED / "labels" / f"{split}.cache"
    if cache.exists():
        cache.unlink()
        print(f"Removed stale cache: {cache}")

total_added = sum(sum(d.values()) for d in added.values())
print(f"\nProcessed {total_processed} images. Test split untouched (honest eval).")
print(f"Total pseudo-labels added: {total_added}")
for split, counts in added.items():
    n = sum(counts.values())
    print(f"  {split:>5s}: {n:>6d} new boxes  {dict(counts)}")
if total_added == 0:
    print("\n!! WARNING: no pseudo-labels added. Either PSEUDO_CONF is too high")
    print("   or the model genuinely doesn't see anything new — bug? Check predictions.")


Per-dataset valid pseudo-classes (unified IDs / names):
  d1: [0, 3]  ['crack', 'mold']
  d2: [0]  ['crack']
  d4: [1]  ['water_leakage']
  d5: [2]  ['electrical_fault']

[train] pseudo-labeling 26379 images at conf>=0.4 ...
  64/26379  added so far: {}
  1664/26379  added so far: {'crack': 41, 'mold': 29}
  3264/26379  added so far: {'crack': 167, 'mold': 54}
  4864/26379  added so far: {'crack': 191, 'mold': 92}
  6464/26379  added so far: {'crack': 213, 'mold': 93}
  8064/26379  added so far: {'crack': 262, 'mold': 130}
  9664/26379  added so far: {'crack': 262, 'mold': 130}
  11264/26379  added so far: {'crack': 262, 'mold': 130}
  12864/26379  added so far: {'crack': 262, 'mold': 130}
  14464/26379  added so far: {'crack': 262, 'mold': 130}
  16064/26379  added so far: {'crack': 262, 'mold': 130}
  17664/26379  added so far: {'crack': 262, 'mold': 130}
  19264/26379  added so far: {'crack': 262, 'mold': 130, 'water_leakage': 2}
  20864/26379  added so far: {'crack': 262, 'mold': 1

In [37]:
# Cell 8 — Train
results = model.train(
    data=str(UNIFIED / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    cache="disk",
    lr0=LR0,
    cos_lr=True,
    warmup_epochs=WARMUP,
    close_mosaic=CLOSE_MOSAIC_EPOCHS,
    mosaic=MOSAIC,
    mixup=MIXUP,
    cls=CLS_GAIN,
    patience=PATIENCE,
    exist_ok=True,
    project=str(RUNS_DIR),
    name="defect_finetune",
    seed=SEED,
    plots=True,
)

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=1.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/unified_dataset/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=35, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/kaggle/input/models/abdelrahmanmajed/2/pytorch/default/1/best (1).pt, m

In [38]:
# Cell 9 — Final eval on test + delta vs baseline
best_new = RUNS_DIR / "defect_finetune" / "weights" / "best.pt"
model = YOLO(str(best_new))

final = model.val(
    data=str(UNIFIED / "data.yaml"),
    split="test",
    imgsz=IMGSZ,
    device=DEVICE,
    augment=True,
    plots=True,
    save_json=False,
    verbose=False,
)
final_per_class = {UNIFIED_CLASSES[i]: float(final.box.maps[i]) for i in range(len(UNIFIED_CLASSES))}
final_map50 = float(final.box.map50)
final_map = float(final.box.map)

print(f"{'':<20s} {'baseline':>10s} {'final':>10s} {'delta':>10s}")
print(f"{'mAP50':<20s} {baseline_map50:>10.4f} {final_map50:>10.4f} {final_map50-baseline_map50:>+10.4f}")
print(f"{'mAP50-95':<20s} {baseline_map:>10.4f} {final_map:>10.4f} {final_map-baseline_map:>+10.4f}")
print()
for cname in UNIFIED_CLASSES:
    b = baseline_per_class[cname]
    f_ = final_per_class[cname]
    print(f"{cname:<20s} {b:>10.4f} {f_:>10.4f} {f_-b:>+10.4f}")

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 90% ━━━━━━━━━━╸─ 201/223 1.6it/s 1:33<13.4sWARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 91% ━━━━━━━━━━╸─ 202/223 1.6it/s 1:34<12.7sWARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 91% ━━━━━━━━━━╸─ 203/223 1.6it/s 1:34<12.1sWARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 91% ━━━━━━━━━━╸─ 204/223 1.6it/s 1:35<11.5sWARNING ⚠️ Model does not support 'augment=True', reverting to single-scale prediction.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 92% ━━━━━━━━━━━─ 205/223 1

In [39]:
# Cell 10 — Save weights to a persistent location
last_new = RUNS_DIR / "defect_finetune" / "weights" / "last.pt"

if IS_KAGGLE:
    out_dir = Path("/kaggle/working")
    shutil.copy2(best_new, out_dir / "best.pt")
    shutil.copy2(last_new, out_dir / "last.pt")
    print(f"Saved to {out_dir}/ — download from Kaggle's Output tab after the session ends.")
else:
    out_dir = Path(r"D:/Graduation project/ML/finetuned")
    out_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best_new, out_dir / "best.pt")
    shutil.copy2(last_new, out_dir / "last.pt")
    print(f"Saved to {out_dir}/")

print(f"\nbest.pt size: {best_new.stat().st_size / 1e6:.1f} MB")

Saved to /kaggle/working/ — download from Kaggle's Output tab after the session ends.

best.pt size: 53.0 MB
